In [ ]:
!pip install pyfaidx transformers datasets tqdm ipywidgets
!pip install transformers accelerate --quiet
!pip -q install "transformers>=4.55" "huggingface_hub>=0.23" safetensors torch requests seaborn matplotlib numpy seam-nn
!pip install --force-reinstall pandas-q
!pip install numpy==1.26.4
!pip install --upgrade scipy
!pip install --upgrade pillow

# !wget -c http://hgdownload.cse.ucsc.edu/goldenpath/hg38/bigZips/hg38.fa.gz
# !gunzip -k hg38.fa.gz

In [ ]:
from huggingface_hub import login

login("") # Replace with your actual token

In [9]:
import argparse
import os
import csv
 
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import random
 
from tqdm import tqdm
from pyfaidx import Fasta
from transformers import AutoModel, AutoModelForMaskedLM, AutoTokenizer

# ──────────────────────────────────────────────
# Config
# ──────────────────────────────────────────────
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
 
GENOME_PATH  = "/gpfs/home/pl2948/evo2/hg38/hg38.fa"
VARIANTS_CSV = "/gpfs/home/pl2948/VEP_eval/ClinVarBenchmark_PB_202602.csv"
OUTPUT_DIR   = "/gpfs/home/pl2948/VEP_eval/ntv3_chunks"
 
CONTEXT_LEN  = {"pre": 131072, "post": 131072}
MODEL_ID     = {
    "pre":  "InstaDeepAI/NTv3_650M_pre",
    "post": "InstaDeepAI/NTv3_650M_post",
}
SPECIES = "human"
 
SCORE_COLS = [
    "idx",
    "#CHROM", "POS", "REF", "ALT",
    "pre_position_llr", "pre_seq_pllr",
    "post_position_llr", "post_seq_pllr", "post_log2fc_max",
]

# ──────────────────────────────────────────────
# Reproducibility
# ──────────────────────────────────────────────
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
        torch.backends.cudnn.deterministic = True
        torch.backends.cudnn.benchmark = False

set_seed(42)

In [10]:
# ──────────────────────────────────────────────
# Sequence helper
# ──────────────────────────────────────────────
def get_ref_alt_seqs(genome, row, context_len):
    try:
        chrom = str(row["#CHROM"])
        if not chrom.startswith("chr"):
            chrom = "chr" + chrom
        pos = int(row["POS"])
        ref = str(row["REF"]).upper()
        alt = str(row["ALT"]).upper()
 
        if len(ref) != 1 or len(alt) != 1:
            return None, None, None
 
        half  = context_len // 2
        start = pos - half
        end   = pos + half
 
        if chrom not in genome:
            return None, None, None
 
        chrom_len = len(genome[chrom])
        seq       = genome[chrom][max(0, start):min(end, chrom_len)].seq.upper()
        seq       = "N" * max(0, -start) + seq + "N" * max(0, end - chrom_len)
 
        if len(seq) != context_len:
            return None, None, None
 
        mut_pos = half - 1
        if seq[mut_pos] != ref:
            return None, None, None
 
        return seq, seq[:mut_pos] + alt + seq[mut_pos + 1:], mut_pos
 
    except Exception as e:
        print(f"  [seq error] {e}")
        return None, None, None
 
 
# ──────────────────────────────────────────────
# Tokenizer helpers
# ──────────────────────────────────────────────
def get_base2id(tokenizer):
    return {b: tokenizer.convert_tokens_to_ids(b) for b in "ACGT"}
 
 
# ──────────────────────────────────────────────
# Single-variant scoring
# ──────────────────────────────────────────────
@torch.no_grad()
def score_variant(model, tokenizer, ref_seq, alt_seq, mut_pos,
                  ref_base, alt_base, base2id,
                  is_post=False, species_ids=None):
    """
    Two forward passes (ref + alt). Intermediate tensors freed immediately.
    Returns dict of scores or None.
    """
    ref_id = base2id.get(ref_base)
    alt_id = base2id.get(alt_base)
    if ref_id is None or alt_id is None:
        return None
 
    valid_ids = torch.tensor(list(base2id.values()))
 
    def forward(seq):
        inputs = tokenizer(
            seq,
            add_special_tokens=False,
            padding=True,
            pad_to_multiple_of=128,
            return_tensors="pt",
            truncation=False,
        )
        input_ids = inputs["input_ids"].to(DEVICE)
        if is_post:
            out = model(input_ids=input_ids,
                        species_ids=species_ids,
                        return_dict=True)
        else:
            out = model(input_ids=input_ids, return_dict=True)
        return out, input_ids[0].cpu()
 
    def mean_logp(logits_1d, input_ids):
        lp       = F.log_softmax(logits_1d, dim=-1)
        true_lp  = lp[torch.arange(len(input_ids)), input_ids]
        is_valid = torch.isin(input_ids, valid_ids)
        return true_lp[is_valid].mean().item()
 
    # ── ref pass ──
    ref_out, ref_ids = forward(ref_seq)
    ref_logits       = ref_out.logits[0].float().cpu()   # (L, V)
 
    # position LLR: raw logit difference at variant position, ref seq only
    pos_logits       = ref_logits[mut_pos]               # (V,)
    position_llr     = (pos_logits[alt_id] - pos_logits[ref_id]).item()
 
    # sequence pseudo-LLR: ref seq mean log P
    ref_mean         = mean_logp(ref_logits, ref_ids)
 
    bw_ref = (ref_out.bigwig_tracks_logits[0].float().cpu()
              if is_post and ref_out.bigwig_tracks_logits is not None
              else None)
 
    del ref_out, ref_logits
    torch.cuda.empty_cache()
 
    # ── alt pass (for seq_pllr only) ──
    alt_out, alt_ids = forward(alt_seq)
    alt_logits       = alt_out.logits[0].float().cpu()
    alt_mean         = mean_logp(alt_logits, alt_ids)
 
    bw_alt = (alt_out.bigwig_tracks_logits[0].float().cpu()
              if is_post and alt_out.bigwig_tracks_logits is not None
              else None)
 
    del alt_out, alt_logits
    torch.cuda.empty_cache()
 
    result = {
        "position_llr": position_llr,
        "seq_pllr":     alt_mean - ref_mean,
    }
 
    if is_post and bw_ref is not None and bw_alt is not None:
        eps    = 1e-6
        log2fc = torch.log2((bw_alt + eps) / (bw_ref + eps))
        L      = log2fc.shape[0]
        s, e   = int(L * 0.3125), int(L * 0.6875)
        result["log2fc_max"] = log2fc[s:e].abs().max().item()
        del bw_ref, bw_alt, log2fc
 
    return result

In [11]:

# ──────────────────────────────────────────────
# Main
# ──────────────────────────────────────────────
def run_chunk(chunk_idx, n_chunks):
    set_seed(42)
    os.makedirs(OUTPUT_DIR, exist_ok=True)
 
    out_path = os.path.join(OUTPUT_DIR, f"ntv3_chunk{chunk_idx}.csv")
 
    # ── Resume: load already-completed idx ──
    done_idxs = set()
    if os.path.exists(out_path):
        try:
            done_df   = pd.read_csv(out_path, usecols=["idx"])
            done_idxs = set(done_df["idx"].tolist())
            print(f"[Resume] {len(done_idxs)} variants already done, skipping.")
        except Exception:
            print("[Resume] Could not read existing file, starting fresh.")
 
    # Open CSV for appending (write header only if file is new/empty)
    file_is_new = not os.path.exists(out_path) or os.path.getsize(out_path) == 0
    csv_file = open(out_path, "a", newline="", buffering=1)  # line-buffered
    writer   = csv.DictWriter(csv_file, fieldnames=SCORE_COLS)
    if file_is_new:
        writer.writeheader()
 
    # ── Load data ──
    df    = pd.read_csv(VARIANTS_CSV, low_memory=False)
    indices = np.array_split(np.arange(len(df)), n_chunks)[chunk_idx]
    chunk   = df.iloc[indices]
    genome = Fasta(GENOME_PATH)
 
    print(f"\n=== Chunk {chunk_idx}/{n_chunks} | "
          f"{len(chunk)} variants | device={DEVICE} ===")
 
    # ── Load pre model ──
    print(f"Loading {MODEL_ID['pre']} ...")
    pre_tok = AutoTokenizer.from_pretrained(
        MODEL_ID["pre"], trust_remote_code=True)
    pre_model = AutoModelForMaskedLM.from_pretrained(
        MODEL_ID["pre"], trust_remote_code=True
    ).to(DEVICE).eval()
    pre_base2id = get_base2id(pre_tok)
 
    # ── Load post model ──
    print(f"Loading {MODEL_ID['post']} ...")
    post_tok = AutoTokenizer.from_pretrained(
        MODEL_ID["post"], trust_remote_code=True)
    post_model = AutoModel.from_pretrained(
        MODEL_ID["post"], trust_remote_code=True
    ).to(DEVICE).eval()
    post_base2id = get_base2id(post_tok)
    post_species = post_model.encode_species([SPECIES]).to(DEVICE)
 
    # ── Score each variant ──
    for idx, row in tqdm(chunk.iterrows(), total=len(chunk),
                         desc=f"chunk{chunk_idx}"):
        if idx in done_idxs:
            continue
 
        ref_base = str(row["REF"]).upper()
        alt_base = str(row["ALT"]).upper()
 
        # pre
        ref_seq, alt_seq, mut_pos = get_ref_alt_seqs(
            genome, row, CONTEXT_LEN["pre"])
        pre_scores = {}
        if ref_seq is not None:
            try:
                res = score_variant(
                    pre_model, pre_tok,
                    ref_seq, alt_seq, mut_pos,
                    ref_base, alt_base, pre_base2id,
                    is_post=False,
                )
                if res is not None:
                    pre_scores = {f"pre_{k}": v for k, v in res.items()}
            except Exception as e:
                print(f"  [pre ERROR row {idx}] {e}")
 
        # post
        ref_seq, alt_seq, mut_pos = get_ref_alt_seqs(
            genome, row, CONTEXT_LEN["post"])
        post_scores = {}
        if ref_seq is not None:
            try:
                res = score_variant(
                    post_model, post_tok,
                    ref_seq, alt_seq, mut_pos,
                    ref_base, alt_base, post_base2id,
                    is_post=True,
                    species_ids=post_species,
                )
                if res is not None:
                    post_scores = {f"post_{k}": v for k, v in res.items()}
            except Exception as e:
                print(f"  [post ERROR row {idx}] {e}")
 
        # write row (even if only one model succeeded)
        if pre_scores or post_scores:
            row_dict = {
                "idx":                  idx,
                "#CHROM":               str(row["#CHROM"]),
                "POS":                  int(row["POS"]),
                "REF":                  str(row["REF"]).upper(),
                "ALT":                  str(row["ALT"]).upper(),
                "pre_position_llr":     pre_scores.get("pre_position_llr",   ""),
                "pre_seq_pllr":         pre_scores.get("pre_seq_pllr",       ""),
                "post_position_llr":    post_scores.get("post_position_llr", ""),
                "post_seq_pllr":        post_scores.get("post_seq_pllr",     ""),
                "post_log2fc_max":      post_scores.get("post_log2fc_max",   ""),
            }
            writer.writerow(row_dict)
 
    csv_file.close()
    print(f"Done. Scores written to {out_path}")

In [ ]:
for i in range(100):
    run_chunk(i, 100)

In [ ]:
import pandas as pd, glob
dfs = [pd.read_csv(f) for f in sorted(glob.glob('ntv3_chunks/ntv3_chunk*.csv'))]
pd.concat(dfs).to_csv('ntv3_all_scores.csv', index=False)
print('Done')

In [ ]:
# # 跑完tokenizer之後直接印出來看
# test_seq = "ACGTACGT"
# tokens = pre_tokenizer(test_seq, add_special_tokens=False)["input_ids"]
# print(tokens)
# print(pre_tokenizer.convert_ids_to_tokens(tokens))

# # 確認第一個位置對應的是A還是special token
# print(f"Token 0: {pre_tokenizer.convert_ids_to_tokens([tokens[0]])}")
# print(f"A token id: {pre_tokenizer.convert_tokens_to_ids('A')}")
# print(f"len(seq)={len(test_seq)}, len(tokens)={len(tokens)}")

In [ ]:
# test_species = post_model.encode_species(["human"])
# print(test_species.shape)

In [1]:
import pandas as pd

version = 'updated'# "balancedgenome" #"balanced" # 'updated' # 2023 #

data = pd.read_csv(f"./clinvar_{version}.csv", low_memory=False)
data

,#CHROM,POS,ID,REF,ALT,ClinVar_label,ClinVar_gold_stars,ClinVarName_refseq_ids,ClinVarName_AAPOS,ClinVarName_AAREF,...,PrimateAI_3D,AlphaMissense,PhyloGPN,Evo2_40B,Rule_based,ntv3_pre_position_llr,ntv3_pre_seq_pllr,ntv3_post_position_llr,ntv3_post_seq_pllr,ntv3_post_log2fc_max
0,1,930165,1164676,G,A,0.0,1.0,NM_001385641.1,207.0,R,...,NaN,NaN,-1.801111,-0.001966,0.357648,1.033955,1.668930e-06,1.666055,-0.000043,2.827893
1,1,930204,1170208,G,A,0.0,2.0,NM_001385641.1,220.0,R,...,NaN,NaN,-0.810768,-0.001260,0.357648,1.569786,-4.708767e-06,1.588054,-0.000701,3.240415
2,1,930285,1165489,G,A,0.0,1.0,NM_001385641.1,247.0,R,...,NaN,NaN,-0.815928,0.000659,0.357648,1.588131,-3.576279e-07,1.680781,-0.000585,3.104318
3,1,930314,1170010,C,T,0.0,2.0,NM_001385641.1,257.0,H,...,NaN,NaN,-1.633337,-0.003418,0.357648,3.150200,-7.659197e-06,3.216790,0.000197,1.822163
4,1,930325,1168187,C,T,0.0,1.0,NM_001385641.1,260.0,I,...,NaN,NaN,-1.464217,-0.000417,0.005252,4.653107,1.162291e-05,3.354657,-0.000510,2.667778
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
242127,Y,56960499,726033,A,G,0.0,1.0,NM_001304990.2,36.0,K,...,NaN,NaN,-0.414166,-0.000091,0.005252,1.494453,5.513430e-06,2.435236,0.001451,2.540560
242128,Y,57084754,777848,C,G,0.0,2.0,NM_005638.6,109.0,V,...,NaN,NaN,-1.074717,-0.000446,0.005252,1.905708,1.013279e-06,1.233571,0.002534,2.924839
242129,Y,57087054,721652,T,C,0.0,2.0,NM_005638.6,127.0,M,...,0.300756,0.08,-3.095034,-0.000956,0.357648,6.131863,4.357100e-05,2.936291,0.000638,2.801300
242130,Y,57190090,785714,G,A,0.0,1.0,NM_002186.3,NaN,NaN,...,NaN,NaN,-0.255677,-0.000129,0.020306,2.886108,-5.185604e-06,-0.014036,0.005069,2.687465


In [2]:
ntv3 = pd.read_csv('ntv3_all_scores.csv', low_memory=False)
ntv3

,idx,#CHROM,POS,REF,ALT,pre_position_llr,pre_seq_pllr,post_position_llr,post_seq_pllr,post_log2fc_max
0,0,1,930165,G,A,1.033955,1.668930e-06,1.666055,-0.000043,2.827893
1,1,1,930204,G,A,1.569786,-4.708767e-06,1.588054,-0.000701,3.240415
2,2,1,930285,G,A,1.588131,-3.576279e-07,1.680781,-0.000585,3.104318
3,3,1,930314,C,T,3.150200,-7.659197e-06,3.216790,0.000197,1.822163
4,4,1,930325,C,T,4.653107,1.162291e-05,3.354657,-0.000510,2.667778
...,...,...,...,...,...,...,...,...,...,...
242127,121095,2,27462090,T,C,2.888035,8.791685e-06,1.415020,0.000005,1.950790
242128,121096,2,27462521,G,A,2.186752,5.364418e-07,0.791272,0.000006,2.571507
242129,121097,2,27462763,G,A,4.880951,5.072355e-05,0.243870,0.000136,3.368960
242130,121098,2,27463147,G,A,4.721272,2.563000e-05,-0.478282,0.000483,5.328032


In [ ]:
# Sort both dataframes by the key columns
key_cols = ["#CHROM", "POS", "REF", "ALT"]

data_sorted = data.sort_values(key_cols).reset_index(drop=True)
ntv3_sorted = ntv3.sort_values(key_cols).reset_index(drop=True)

# Verify if order matches after sorting
keys_match = (data_sorted[key_cols].values == ntv3_sorted[key_cols].values).all()
print(f"Keys match after sorting: {keys_match}")
print(f"data rows: {len(data_sorted)}, ntv3 rows: {len(ntv3_sorted)}")

if keys_match and len(data_sorted) == len(ntv3_sorted):
    score_cols = ["pre_position_llr", "pre_seq_pllr",
                  "post_position_llr", "post_seq_pllr", "post_log2fc_max"]
    for col in score_cols:
        data_sorted[f"ntv3_{col}"] = ntv3_sorted[col].values
else:
    # fallback to left join
    ntv3_rename = ntv3[key_cols + score_cols].rename(
        columns={c: f"ntv3_{c}" for c in score_cols}
    )
    data_sorted = data_sorted.merge(ntv3_rename, on=key_cols, how="left")

In [7]:
data_sorted.to_csv("./clinvar_updated.csv", index=False)